In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


In [13]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Paths to your .npz files
TRAIN_PATH = "processed_data/train_100_max_frame.npz"
VAL_PATH   = "processed_data/val_100_max_frame.npz"
TEST_PATH  = "processed_data/test_100_max_frame.npz"

def load_split(npz_path):
    """
    Loads a split from an .npz file and returns:
      X    : np.ndarray, shape = (n_samples, max_frame, 126)
      mask : np.ndarray, shape = (n_samples, max_frame)
      y    : list of string labels, length = n_samples
    """
    data = np.load(npz_path, allow_pickle=True)
    X    = data["X"]          # shape: (n_samples, max_frame, 126)
    mask = data["mask"]       # shape: (n_samples, max_frame)
    y    = data["y"].tolist() # list of strings
    return X, mask, y

# 1) Load raw arrays, masks, + string labels
X_train, mask_train, y_train_str = load_split(TRAIN_PATH)
X_val,   mask_val,   y_val_str   = load_split(VAL_PATH)
X_test,  mask_test,  y_test_str  = load_split(TEST_PATH)

print("Raw shapes:")
print(f"  X_train:    {X_train.shape},   # samples = {len(y_train_str)}")
print(f"  mask_train: {mask_train.shape}  (# frames per sample = {mask_train.shape[1]})")
print(f"  X_val:      {X_val.shape},     # samples = {len(y_val_str)}")
print(f"  mask_val:   {mask_val.shape}    (# frames per sample = {mask_val.shape[1]})")
print(f"  X_test:     {X_test.shape},    # samples = {len(y_test_str)}")
print(f"  mask_test:  {mask_test.shape}   (# frames per sample = {mask_test.shape[1]})")

Raw shapes:
  X_train:    (2124, 100, 126),   # samples = 2124
  mask_train: (2124, 100)  (# frames per sample = 100)
  X_val:      (277, 100, 126),     # samples = 277
  mask_val:   (277, 100)    (# frames per sample = 100)
  X_test:     (416, 100, 126),    # samples = 416
  mask_test:  (416, 100)   (# frames per sample = 100)


In [14]:
# 2) Encode string labels → integer indices
le = LabelEncoder()
# Fit on all labels (train+val+test) so that index mapping is consistent:
all_labels = y_train_str + y_val_str + y_test_str
le.fit(all_labels)

y_train = le.transform(y_train_str)  # integer array, shape = (n_train,)
y_val   = le.transform(y_val_str)    # shape = (n_val,)
y_test  = le.transform(y_test_str)   # shape = (n_test,)

num_classes = len(le.classes_)
print(f"Detected {num_classes} distinct labels.")


Detected 93 distinct labels.


In [15]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers, callbacks
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [34]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
T_MAX      = 100     # number of frames per example
D_FEATURE  = 126    # 21 keypoints × 2 hands × 3 coords
NUM_CLASSES = 93    # number of distinct sign labels
BATCH_SIZE  = 32
EPOCHS      = 80
LEARNING_RATE = 1e-3
CLIP_NORM     = 1.0
# ──────────────────────────────────────────────────────────────────────────────


def build_sign_model():
    model = models.Sequential([
        # 1) Masking to ignore zero-padded frames
        layers.Input(shape=(T_MAX, D_FEATURE)),

        layers.Conv1D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv1D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.1),
        layers.Masking(mask_value=0.0, input_shape=(T_MAX, D_FEATURE)),
        # 2) First Bi-LSTM layer (return full sequence)
        layers.Bidirectional(
            layers.LSTM(
                units=128,
                return_sequences=True,
                dropout=0.1,            # very light dropout
                recurrent_dropout=0   # no recurrent dropout
                # kernel_regularizer removed
            )
        ),

        # 3) Second Bi-LSTM layer (return last hidden state)
        layers.Bidirectional(
            layers.LSTM(
                units=128,
                return_sequences=False,
                dropout=0.1,            # very light dropout
                recurrent_dropout=0   # no recurrent dropout
                # kernel_regularizer removed
            )
        ),

        # 4) Dense head with minimal regularization
        layers.Dense(
            units=128,
            activation="relu"
            # kernel_regularizer removed
        ),
        layers.Dropout(0.1),   # small dropout
        layers.BatchNormalization(),

        # 5) Final softmax
        layers.Dense(units=NUM_CLASSES, activation="softmax")
    ])
    return model



# 1) Build & compile:
model = build_sign_model()
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=10,
    min_delta=0.001,
    restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

history = model.fit(
    x=X_train,         # (N_train, max_frame, 126)
    y=y_train,         # (N_train,)
    batch_size=32,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 21s 193ms/step - accuracy: 0.0212 - loss: 4.5992 - val_accuracy: 0.0325 - val_loss: 4.2937 - learning_rate: 0.0010
Epoch 2/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 19s 286ms/step - accuracy: 0.0738 - loss: 3.9948 - val_accuracy: 0.0722 - val_loss: 4.0154 - learning_rate: 0.0010
Epoch 3/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 24s 362ms/step - accuracy: 0.0863 - loss: 3.7105 - val_accuracy: 0.0397 - val_loss: 4.2315 - learning_rate: 0.0010
Epoch 4/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 23s 350ms/step - accuracy: 0.1153 - loss: 3.5597 - val_accuracy: 0.0830 - val_loss: 3.7500 - learning_rate: 0.0010
Epoch 5/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 24s 361ms/step - accuracy: 0.1338 - loss: 3.3961 - val_accuracy: 0.0469 - val_loss: 4.3991 - learning_rate: 0.0010
Epoch 6/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 23s 347ms/step - accuracy: 0.1464 - loss: 3.3695 - val_accuracy: 0.0903 - val_loss: 4.0618 - learning_rate: 0.0010
Epoch 7/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 23s 348ms/step - accuracy: 0.1823 - l

Best so far: training acc 0.6985, val acc 0.42

In [35]:
# ── Inference & Metrics ──────────────────────────────────────────────────────

# 3) Evaluate on the held-out test set (this prints loss & accuracy):
test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE)
print(f"\nTest loss: {test_loss:.4f}   |   Test accuracy: {test_acc:.4f}")

# 4) Predict class probabilities on X_test:
y_proba = model.predict(X_test, batch_size=BATCH_SIZE)  # shape: (n_test, NUM_CLASSES)

# Convert to integer‐class predictions:
y_pred = np.argmax(y_proba, axis=1)  # shape: (n_test,)

# 5) If you want a human-readable classification report, decode labels back to strings:
#    Assume you used a LabelEncoder 'le' (fitted on all labels) before training:
#        le = LabelEncoder()
#        le.fit(all_labels)  # where all_labels = y_train_str + y_val_str + y_test_str
#    And then y_train = le.transform(y_train_str), etc.
#    Now invert:
le = LabelEncoder()
# (Re‐fit on all string labels just to recover the mapping here:)
all_str_labels = y_train_str + y_val_str + y_test_str
le.fit(all_str_labels)

y_test_str_decoded = le.inverse_transform(y_test)   # length = n_test
y_pred_str_decoded = le.inverse_transform(y_pred)   # length = n_test

# 6) Print a full classification report:
print("\nClassification Report (per‐class):")
print(
    classification_report(
        y_true=y_test_str_decoded,
        y_pred=y_pred_str_decoded,
        digits=4
    )
)

# 7) (Optional) Show confusion matrix as well:
cm = confusion_matrix(y_test_str_decoded, y_pred_str_decoded, labels=le.classes_)
print("\nConfusion Matrix (rows=true, cols=predicted):")
print(cm)

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - accuracy: 0.4772 - loss: 2.2326

Test loss: 2.3012   |   Test accuracy: 0.4688
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step

Classification Report (per‐class):
              precision    recall  f1-score   support

       about     0.0000    0.0000    0.0000         1
       again     0.6667    0.5000    0.5714         8
         ask     0.0000    0.0000    0.0000         1
         bad     0.5000    0.2500    0.3333         4
         boy     0.5714    0.8889    0.6957         9
         buy     0.1667    0.5000    0.2500         2
         can     0.1818    1.0000    0.3077         2
        come     0.0000    0.0000    0.0000         1
   different     0.6667    0.5000    0.5714         4
       drink     0.6667    0.4000    0.5000         5
        easy     0.5000    1.0000    0.6667         1
         eat     0.2857    0.6667    0.4000         3
      family     0.7500    0.3750    0.5000         8
        feel     1.0000    0.6667    0.8000   

/Users/Wilson/miniconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/Wilson/miniconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/Wilson/miniconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/Wilson/m

In [37]:
model.save("sign_model_test_47.keras")